# Save, load and export

A trained model is stored in one of three formats, each for a different use:

| Format | Methods | Use it to |
|---|---|---|
| *nnodely* folder | `save`, `Modely.load` | keep working on the model with *nnodely*: train it more, compose it, export it later |
| Keras file, `.keras` | `export_keras`, `Modely.import_keras` | run the network as a plain `keras.Model`, without *nnodely* |
| ONNX file, `.onnx` | `export_onnx`, `Modely.validate_onnx` | deploy it outside Python, with any ONNX runtime |

This tutorial saves and loads a model made of blocks, exports it to Keras and to
ONNX, and runs every copy on the same data to check that they agree.

In [1]:
import os

# Pick the Keras backend before nnodely (or Keras) is imported:
# "tensorflow" (the default), "torch" or "jax".
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import tempfile
from pathlib import Path

import keras
import numpy as np

from nnodely import (
    DataLoader,
    Fir,
    Input,
    Linear,
    Modely,
    Output,
    Parameter,
    Tanh,
    set_seed,
)

set_seed(0)
workdir = Path(tempfile.mkdtemp(prefix="nnodely_save_"))


def tree(folder):
    """Print the files of a folder, indented by depth."""
    folder = Path(folder)
    for path in sorted(folder.rglob("*")):
        depth = len(path.relative_to(folder).parts) - 1
        print("    " * depth + path.name + ("/" if path.is_dir() else ""))


def to_numpy(results):
    return {name: keras.ops.convert_to_numpy(value) for name, value in results.items()}

## A model to store

A `filter` block, trained first on its own, then used by a `controller` that
adds a small network and a trained gain.

In [2]:
rng = np.random.default_rng(0)
T = 2000
signal = np.repeat(rng.uniform(-1.0, 1.0, T // 20), 20)
smooth = np.convolve(signal, np.ones(8) / 8.0)[:T]

raw = Input("raw", tw=8)
filtered = Output("filtered", Fir(features=1, name="smoother")([raw]))
filter_block = Modely("filter", inputs=[raw], outputs=[filtered])
filter_block.minimize("smooth", filtered, Input("smooth", tw=1))
filter_block.build()
filter_block.train(
    DataLoader(filter_block, source={"raw": signal, "smooth": smooth}),
    epochs=20,
    batch_size=32,
    lr=1e-2,
    printer=None,
)

sensor = Input("sensor", tw=8)
gain = Parameter("gain", value=2.0)
command = Output("command", gain * Tanh()(Linear(features=1)([filter_block([sensor])])))
controller = Modely("controller", inputs=[sensor], outputs=[command]).build()

sample = {"sensor": rng.uniform(-1.0, 1.0, (4, 8, 1)).astype("float32")}
reference = to_numpy(controller(sample))
reference["command"].ravel()

array([-1.2607837 , -1.0391294 , -1.1817975 , -0.97956115], dtype=float32)

## Saving and loading

`save(path)` writes a folder: the graph in `model.json`, the weights in
`model.weights.h5`, and every model it calls as a block - or rolls out in a
`Loop` - saved the same way in a sub-folder named after it.

In [3]:
controller.save(workdir / "controller")
tree(workdir / "controller")

filter/
    model.json
    model.weights.h5
model.json
model.weights.h5


`Modely.load(path)` gives back a complete `Modely`, built and with its weights:
it computes what the saved one did.

In [4]:
restored = Modely.load(workdir / "controller")
print(np.allclose(to_numpy(restored(sample))["command"], reference["command"]))
restored.models

True


┏━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ idx ┃ model  ┃ inputs ┃ outputs  ┃ parameters ┃ built ┃
┡━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ 0   │ filter │ raw    │ filtered │ 2          │ yes   │
└─────┴────────┴────────┴──────────┴────────────┴───────┘

The loaded model is a model of its own: its weights are not shared with the
model it was saved from, nor with another load of the same file. It lists the
same blocks, parameters and inputs, picked by name or index as before.

Each sub-folder is itself a saved model, so a trained block can be taken out of
the model that holds it:

In [5]:
block = Modely.load(workdir / "controller" / "filter")
block

┏━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ model      ┃ filter   ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━┩
│ inputs     │ raw      │
│ outputs    │ filtered │
│ minimizers │ -        │
└────────────┴──────────┘

### Objectives are not saved

Objectives belong to a training session, not to the model: a loaded model has
none. Declare them again, build, and training continues from the loaded
weights:

In [6]:
print("minimizers after load:", list(restored.minimizers))

target = Input("target", tw=1)
restored.minimize("track", "command", target)
restored.build()
train_data = DataLoader(restored, source={"sensor": signal, "target": 0.5 * smooth})
history = restored.train(train_data, epochs=5, batch_size=32, lr=1e-3, printer=None)
print("loss:", [round(value, 5) for value in history["loss"]])

minimizers after load: []
loss: [1.63428, 1.11754, 0.66488, 0.32396, 0.16033]


### Architecture only

`save(path, weights=False)` stores the graph alone - to train the same model
elsewhere - and `load(path, weights=False)` ignores saved weights. Either way
the loaded model starts from the initial values `build()` gives it.

In [7]:
controller.save(workdir / "architecture", weights=False)
tree(workdir / "architecture")
fresh = Modely.load(workdir / "architecture")
print(
    "same as the trained one:",
    np.allclose(to_numpy(fresh(sample))["command"], reference["command"]),
)

filter/
    model.json
model.json
same as the trained one: False


## Keras

`export_keras(folder)` writes `<folder>/<model name>.keras`: a stand-alone copy
of the network for inference. `Modely.import_keras` (or `keras.saving.load_model`)
reads it back as a plain `keras.Model`, which runs on any Keras backend and is
called with a dict of batched float32 arrays:

In [8]:
controller.export_keras(workdir / "exports")
keras_model = Modely.import_keras(workdir / "exports" / "controller.keras")
print(type(keras_model).__name__)
print(np.allclose(to_numpy(keras_model(sample))["command"], reference["command"]))

Functional
True


The `.keras` file is made for inference. A weight that *nnodely* reads from
several places - a block also used inside a `Loop`, a `Gradient` - is written
once per place, all with the same value: trained further as a plain Keras model
those copies would drift apart. To keep training, use `save` and `load`.

## ONNX

ONNX is the format to deploy a network outside Python - in C++, on an embedded
board, in another framework. The export needs the `onnx` extra
(`pip install "nnodely[onnx]"`) and the **TensorFlow** or **PyTorch** backend:
JAX cannot export ONNX.

In [9]:
backend = keras.backend.backend()
print("backend:", backend)

if backend in ("tensorflow", "torch"):
    controller.export_onnx(workdir / "exports")
    onnx_file = workdir / "exports" / "controller.onnx"
    outputs = Modely.validate_onnx(onnx_file, sample, return_dict=True)
    print(np.allclose(outputs["command"], reference["command"], atol=1e-5))

backend: tensorflow
True


`validate_onnx` runs the file with ONNX Runtime, to check it against the model.
In deployment the file is run by a runtime directly. The inputs and outputs keep
their names, and the arrays their layout, `(batch, *S, T, *D)`:

In [10]:
if backend in ("tensorflow", "torch"):
    import onnxruntime as ort

    session = ort.InferenceSession(str(onnx_file))
    for port in session.get_inputs():
        print("input ", port.name, port.shape)
    for port in session.get_outputs():
        print("output", port.name, port.shape)
    (command_out,) = session.run(None, sample)
    print(command_out.ravel())

input  sensor ['unk__46', 8, 1]
output command ['unk__47', 'unk__48', 1]
[-1.2607836  -1.0391293  -1.1817975  -0.97956115]


`export_onnx(folder, batch_size=n)` fixes the batch axis, and
`input_signature` every input shape, for runtimes that need static shapes.

### What each format supports

| The model contains | `save` / `load` | `export_keras` | ONNX, TensorFlow | ONNX, PyTorch |
|---|---|---|---|---|
| layers, `Parameter`, `Constant`, blocks | yes | yes | yes | yes |
| `Loop` | yes | yes | yes | yes |
| `ModelyLoop` (states fed back, `Integrate`, `Buffer`, ...) | yes | yes | yes, any horizon | a fixed horizon |
| `Gradient` with respect to an input | yes | yes | yes, batch of 1 | no |

## Summary

| Task | Code |
|---|---|
| Save, with its blocks | `model.save("folder")` |
| Load, or one of its blocks | `Modely.load("folder")`, `Modely.load("folder/block")` |
| Train a loaded model | `minimize(...)`, `build()`, `train(...)` |
| Architecture only | `save(..., weights=False)`, `load(..., weights=False)` |
| Keras | `model.export_keras("dir")`, `Modely.import_keras("dir/name.keras")` |
| ONNX | `model.export_onnx("dir")`, `Modely.validate_onnx(file, inputs, return_dict=True)` |

Next: [Parameter and Constant](06_parameter_constant.ipynb).